# Test Alignment of XArrays

In [2]:
import xarray as xr
import numpy as np

rng = np.random.default_rng(1)

x = 3
y = 2
da1 = xr.DataArray(np.zeros((x, y)), {"x": np.arange(x), "y": np.arange(y)})
da1

<xarray.DataArray (x: 3, y: 2)> Size: 48B
array([[0., 0.],
       [0., 0.],
       [0., 0.]])
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 16B 0 1

In [2]:
da1 + xr.DataArray(np.ones((2, 1)), {"x": [1, 2], "y": [1]})

<xarray.DataArray (x: 2, y: 1)> Size: 16B
array([[1.],
       [1.]])
Coordinates:
  * x        (x) int64 16B 1 2
  * y        (y) int64 8B 1

## Grouping by non-dim coordinates

In [3]:
da = xr.DataArray(
    np.zeros((3, 2, 5)), {"x": np.arange(3), "y": np.arange(2), "samples": np.arange(5)}
)
da = da.assign_coords({"z": ("samples", [0, 0, 1, 1, 3])})
da

<xarray.DataArray (x: 3, y: 2, samples: 5)> Size: 240B
array([[[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]]])
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 16B 0 1
  * samples  (samples) int64 40B 0 1 2 3 4
    z        (samples) int64 40B 0 0 1 1 3

In [4]:
da.groupby("z").sum()

<xarray.DataArray (x: 3, y: 2, z: 3)> Size: 144B
array([[[0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.]]])
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 16B 0 1
  * z        (z) int64 24B 0 1 3

In [5]:
import pandas as pd

da = da.assign_coords(
    {"time": ("samples", pd.date_range("2000-01-01", "2000-02-01", periods=5))}
)
da

<xarray.DataArray (x: 3, y: 2, samples: 5)> Size: 240B
array([[[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]]])
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 16B 0 1
  * samples  (samples) int64 40B 0 1 2 3 4
    z        (samples) int64 40B 0 0 1 1 3
    time     (samples) datetime64[ns] 40B 2000-01-01 ... 2000-02-01

In [6]:
da = da.swap_dims(samples="time")
da

<xarray.DataArray (x: 3, y: 2, time: 5)> Size: 240B
array([[[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]]])
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 16B 0 1
    samples  (time) int64 40B 0 1 2 3 4
    z        (time) int64 40B 0 0 1 1 3
  * time     (time) datetime64[ns] 40B 2000-01-01 ... 2000-02-01

In [7]:
da.xindexes

Indexes:
    x        PandasIndex
    y        PandasIndex
    time     PandasIndex

In [8]:
da.dims

('x', 'y', 'time')

In [9]:
da.coords

Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 16B 0 1
    samples  (time) int64 40B 0 1 2 3 4
    z        (time) int64 40B 0 0 1 1 3
  * time     (time) datetime64[ns] 40B 2000-01-01 ... 2000-02-01

In [10]:
# da.drop_vars(["samples", "z"])

In [11]:
# NOTE: Does not work because 'time' is not a dimension
# da.resample({"time": "D"}).interpolate()
da.resample({"time": "D"}).interpolate()

# Not working somehow?
# da.resample({"time": "1D"}).nearest(tolerance="3D")
# da.resample({"time": "1D"}).mean()

<xarray.DataArray (x: 3, y: 2, time: 32)> Size: 2kB
array([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]])
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 16B 0 1
  * time     (time) datetime64[ns] 256B 2000-01-01 2000-01-02 ... 2000-02-01

In [12]:
dates = pd.date_range("2000-01-01", periods=9, freq="1D").to_list() + [
    pd.Timestamp("2001-01-10")
]
arr = xr.DataArray(
    np.ones((2, 10)),
    dims=("x", "y"),
    coords={"x": [0, 1], "time": ("y", dates)},
)
arr

<xarray.DataArray (x: 2, y: 10)> Size: 160B
array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])
Coordinates:
  * x        (x) int64 16B 0 1
    time     (y) datetime64[ns] 80B 2000-01-01 2000-01-02 ... 2001-01-10
Dimensions without coordinates: y

In [13]:
arr.resample({"time": "1w"}, closed="left", label="left").reduce(np.mean)

<xarray.DataArray (x: 2, time: 55)> Size: 880B
array([[ 1.,  1.,  1., nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan,  1.],
       [ 1.,  1.,  1., nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan,  1.]])
Coordinates:
  * x        (x) int64 16B 0 1
  * time     (time) datetime64[ns] 440B 1999-12-26 2000-01-02 ... 2001-01-07

In [14]:
arr.groupby(arr["time"].dt.year).reduce(np.mean)

<xarray.DataArray (x: 2, year: 2)> Size: 32B
array([[1., 1.],
       [1., 1.]])
Coordinates:
  * x        (x) int64 16B 0 1
  * year     (year) int64 16B 2000 2001

In [15]:
arr.resample({"time": "1h"}, closed="left", label="left").mean()

<xarray.DataArray (x: 2, time: 9001)> Size: 144kB
array([[ 1., nan, nan, ..., nan, nan,  1.],
       [ 1., nan, nan, ..., nan, nan,  1.]])
Coordinates:
  * x        (x) int64 16B 0 1
  * time     (time) datetime64[ns] 72kB 2000-01-01 ... 2001-01-10

## Multiplying datasets

In [16]:
import xarray as xr

ds1 = xr.Dataset(
    {
        "zero": xr.DataArray(
            np.zeros((3, 3)), coords={"x": np.arange(3), "y": np.arange(3)}
        ),
        "one": xr.DataArray(
            np.ones((3, 3)), coords={"x": np.arange(3), "y": np.arange(3)}
        ),
        "two": xr.DataArray(
            np.ones((3, 3)), coords={"x": np.arange(3), "y": np.arange(3)}
        ),
    },
)

ds2 = xr.Dataset(
    {
        "zero": xr.DataArray(
            np.zeros((2, 2)), coords={"x": np.arange(2), "y": np.arange(2)}
        ),
        "one": xr.DataArray(
            np.ones((2, 2)) * 2.0, coords={"x": np.arange(2), "y": np.arange(2)}
        ),
        "three": xr.DataArray(
            np.ones((2, 2)) * 2.0, coords={"x": np.arange(2), "y": np.arange(2)}
        ),
    },
)

In [17]:
ds1

<xarray.Dataset> Size: 264B
Dimensions:  (x: 3, y: 3)
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 24B 0 1 2
Data variables:
    zero     (x, y) float64 72B 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0
    one      (x, y) float64 72B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0
    two      (x, y) float64 72B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0

In [18]:
ds2

<xarray.Dataset> Size: 128B
Dimensions:  (x: 2, y: 2)
Coordinates:
  * x        (x) int64 16B 0 1
  * y        (y) int64 16B 0 1
Data variables:
    zero     (x, y) float64 32B 0.0 0.0 0.0 0.0
    one      (x, y) float64 32B 2.0 2.0 2.0 2.0
    three    (x, y) float64 32B 2.0 2.0 2.0 2.0

In [19]:
ds1 * ds2

<xarray.Dataset> Size: 96B
Dimensions:  (x: 2, y: 2)
Coordinates:
  * x        (x) int64 16B 0 1
  * y        (y) int64 16B 0 1
Data variables:
    zero     (x, y) float64 32B 0.0 0.0 0.0 0.0
    one      (x, y) float64 32B 2.0 2.0 2.0 2.0

In [20]:
ds1 * ds2["three"]

<xarray.Dataset> Size: 128B
Dimensions:  (x: 2, y: 2)
Coordinates:
  * x        (x) int64 16B 0 1
  * y        (y) int64 16B 0 1
Data variables:
    zero     (x, y) float64 32B 0.0 0.0 0.0 0.0
    one      (x, y) float64 32B 2.0 2.0 2.0 2.0
    two      (x, y) float64 32B 2.0 2.0 2.0 2.0

Datasets with differently named variables are not combined!

In [ ]:
ds1["zero"].to_dataset() * ds2["three"].to_dataset()  #!!

<xarray.Dataset> Size: 32B
Dimensions:  (x: 2, y: 2)
Coordinates:
  * x        (x) int64 16B 0 1
  * y        (y) int64 16B 0 1
Data variables:
    *empty*

In [21]:
dt = xr.DataTree.from_dict({"a": ds1, "b": ds2})
dt

<xarray.DataTree>
Group: /
├── Group: /a
│       Dimensions:  (x: 3, y: 3)
│       Coordinates:
│         * x        (x) int64 24B 0 1 2
│         * y        (y) int64 24B 0 1 2
│       Data variables:
│           zero     (x, y) float64 72B 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0
│           one      (x, y) float64 72B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0
│           two      (x, y) float64 72B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0
└── Group: /b
        Dimensions:  (x: 2, y: 2)
        Coordinates:
          * x        (x) int64 16B 0 1
          * y        (y) int64 16B 0 1
        Data variables:
            zero     (x, y) float64 32B 0.0 0.0 0.0 0.0
            one      (x, y) float64 32B 2.0 2.0 2.0 2.0
            three    (x, y) float64 32B 2.0 2.0 2.0 2.0

In [22]:
# xr.DataTree.from_dict(
#     {"/a": ds1, "/b": ds2, "/a/aa": ds1.sel(x=slice(0, 1), y=slice(0, 1))}
# )


In [23]:
xr.combine_by_coords(
    [
        xr.Dataset(
            {
                "zero": xr.DataArray(
                    [[0, 0, 0], [0, 0, np.nan], [0, np.nan, np.nan]],
                    coords={"x": np.arange(3), "y": np.arange(3)},
                ),
                "one": xr.DataArray(
                    np.ones((3, 3)), coords={"x": np.arange(3), "y": np.arange(3)}
                ),
            },
        ),
        xr.Dataset(
            {
                "zero": xr.DataArray(
                    [[np.nan, np.nan, np.nan], [np.nan, np.nan, 0], [np.nan, 0, 0]],
                    coords={"x": np.arange(3), "y": np.arange(3)},
                ),
                # "three": xr.DataArray(
                #     np.ones((2, 2)) * 2.0,
                #     coords={"x": np.arange(3, 5), "y": np.arange(3, 5)},
                # ),
            },
        ),
    ],
    # compat="broadcast_equals",
    # data_vars="all",
    # coords="all",
    # join="outer",
)

<xarray.Dataset> Size: 192B
Dimensions:  (x: 3, y: 3)
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 24B 0 1 2
Data variables:
    zero     (x, y) float64 72B 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0
    one      (x, y) float64 72B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0

## Ragged dimensions

Will be broadcast automatically

In [24]:
da = xr.DataArray(
    np.zeros((3, 3, 3)), coords={dim: np.arange(3) for dim in ("x", "y", "z")}
)
da

<xarray.DataArray (x: 3, y: 3, z: 3)> Size: 216B
array([[[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]]])
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 24B 0 1 2
  * z        (z) int64 24B 0 1 2

In [25]:
da_concat = xr.concat([da.sel(z=slice(0, 1)), da.sel(z=slice(1, 3))], dim="samples")
da_concat

<xarray.DataArray (samples: 2, x: 3, y: 3, z: 3)> Size: 432B
array([[[[ 0.,  0., nan],
         [ 0.,  0., nan],
         [ 0.,  0., nan]],

        [[ 0.,  0., nan],
         [ 0.,  0., nan],
         [ 0.,  0., nan]],

        [[ 0.,  0., nan],
         [ 0.,  0., nan],
         [ 0.,  0., nan]]],


       [[[nan,  0.,  0.],
         [nan,  0.,  0.],
         [nan,  0.,  0.]],

        [[nan,  0.,  0.],
         [nan,  0.,  0.],
         [nan,  0.,  0.]],

        [[nan,  0.,  0.],
         [nan,  0.,  0.],
         [nan,  0.,  0.]]]])
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 24B 0 1 2
  * z        (z) int64 24B 0 1 2
Dimensions without coordinates: samples

In [26]:
da_concat.sel(samples=0)

<xarray.DataArray (x: 3, y: 3, z: 3)> Size: 216B
array([[[ 0.,  0., nan],
        [ 0.,  0., nan],
        [ 0.,  0., nan]],

       [[ 0.,  0., nan],
        [ 0.,  0., nan],
        [ 0.,  0., nan]],

       [[ 0.,  0., nan],
        [ 0.,  0., nan],
        [ 0.,  0., nan]]])
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 24B 0 1 2
  * z        (z) int64 24B 0 1 2

## Datatree and Datasets

In [4]:
ds = xr.Dataset(
    {"var": (["x", "y"], np.ones((3, 4), dtype="float"))},
    coords={"x": np.arange(3), "y": np.arange(4)},
)
data = {
    "/a": ds.sel(x=slice(0, 1)),
    "/b/1": ds.sel(x=slice(2, 3), y=slice(0, 1)),
    "/b/2": ds.sel(x=slice(2, 3), y=slice(2, 4)),
}
dt = xr.DataTree.from_dict(data)
dt

<xarray.DataTree>
Group: /
├── Group: /a
│       Dimensions:  (x: 2, y: 4)
│       Coordinates:
│         * x        (x) int64 16B 0 1
│         * y        (y) int64 32B 0 1 2 3
│       Data variables:
│           var      (x, y) float64 64B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0
└── Group: /b
    ├── Group: /b/1
    │       Dimensions:  (x: 1, y: 2)
    │       Coordinates:
    │         * x        (x) int64 8B 2
    │         * y        (y) int64 16B 0 1
    │       Data variables:
    │           var      (x, y) float64 16B 1.0 1.0
    └── Group: /b/2
            Dimensions:  (x: 1, y: 2)
            Coordinates:
              * x        (x) int64 8B 2
              * y        (y) int64 16B 2 3
            Data variables:
                var      (x, y) float64 16B 1.0 1.0

In [9]:
tree = xr.DataTree()
tree.ds = None
tree.is_empty

True

In [16]:
tree = xr.DataTree.from_dict({"/a": ds})
for path, node in tree.subtree_with_keys:
    print(path)
    print(node.ds)
    print(node.is_empty)
    print(node.has_data)

.
<xarray.DatasetView> Size: 0B
Dimensions:  ()
Data variables:
    *empty*
True
False
a
<xarray.DatasetView> Size: 152B
Dimensions:  (x: 3, y: 4)
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 32B 0 1 2 3
Data variables:
    var      (x, y) float64 96B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0
False
True


In [21]:
tree_zero = tree * xr.zeros_like(ds)
for path, node in tree_zero.subtree_with_keys:
    print(path)
    print(node.ds)
    print(node.is_empty)
    print(node.has_data)

.
<xarray.DatasetView> Size: 56B
Dimensions:  (x: 3, y: 4)
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 32B 0 1 2 3
Data variables:
    *empty*
False
True
a
<xarray.DatasetView> Size: 152B
Dimensions:  (x: 3, y: 4)
Coordinates:
  * x        (x) int64 24B 0 1 2
  * y        (y) int64 32B 0 1 2 3
Data variables:
    var      (x, y) float64 96B 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0
False
True


In [ ]:
res = dt * xr.zeros_like(ds)
res

ValueError: group '/a' is not aligned with its parents:
Group:
    Dimensions:  (x: 2, y: 4)
    Coordinates:
      * x        (x) int64 16B 0 1
      * y        (y) int64 32B 0 1 2 3
    Data variables:
        var      (x, y) float64 64B 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0
From parents:
    Dimensions:  (x: 3, y: 4)
    Coordinates:
      * x        (x) int64 24B 0 1 2
      * y        (y) int64 32B 0 1 2 3

In [53]:
res = xr.DataTree.from_dict(
    {key: dset * xr.zeros_like(ds) for key, dset in data.items()}
)
res

<xarray.DataTree>
Group: /
├── Group: /a
│       Dimensions:  (x: 2, y: 4)
│       Coordinates:
│         * x        (x) int64 16B 0 1
│         * y        (y) int64 32B 0 1 2 3
│       Data variables:
│           var      (x, y) float64 64B 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0
└── Group: /b
    ├── Group: /b/1
    │       Dimensions:  (x: 1, y: 2)
    │       Coordinates:
    │         * x        (x) int64 8B 2
    │         * y        (y) int64 16B 0 1
    │       Data variables:
    │           var      (x, y) float64 16B 0.0 0.0
    └── Group: /b/2
            Dimensions:  (x: 1, y: 2)
            Coordinates:
              * x        (x) int64 8B 2
              * y        (y) int64 16B 2 3
            Data variables:
                var      (x, y) float64 16B 0.0 0.0

In [ ]:
import numpy as np
from functools import partial

# dt.map_over_datasets(partial(xr.align, ds, join="outer"))[0]

<xarray.DataTree>
Group: /
│   Dimensions:  (x: 3, y: 4)
│   Coordinates:
│     * x        (x) int64 24B 0 1 2
│     * y        (y) int64 32B 0 1 2 3
│   Data variables:
│       var      (x, y) float64 96B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0
├── Group: /a
│       Dimensions:  (x: 3, y: 4)
│       Data variables:
│           var      (x, y) float64 96B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0
└── Group: /b
    │   Dimensions:  (x: 3, y: 4)
    │   Data variables:
    │       var      (x, y) float64 96B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0
    ├── Group: /b/1
    │       Dimensions:  (x: 3, y: 4)
    │       Data variables:
    │           var      (x, y) float64 96B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0
    └── Group: /b/2
            Dimensions:  (x: 3, y: 4)
            Data variables:
                var      (x, y) float64 96B 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0